# Iteration 4: five-seed holdout benchmark and Optuna tuning
70/30 stratified splits; 3-fold CV inside training only. Existing ordinal codes are retained. Four selector/scaling benchmarks and tuned RFE20/standardization versions of all five model families. Local `.venv` links to the existing Experimental environment. No installation cell is required. On Colab/Kaggle, upload this project and use an environment with the listed dependencies. Google Drive mounting is optional and unchanged.

In [ ]:
from pathlib import Path
import os, sys
# Set this to your uploaded Iteration 4 directory when using Colab/Kaggle.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'holdout_workflow.py').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'iteration4_BDAS'
assert (PROJECT_ROOT / 'holdout_workflow.py').is_file(), 'Set PROJECT_ROOT to the project folder'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from importlib.metadata import version
for package in ['optuna','scikit-learn','xgboost','lightgbm','numpy','pandas']:
    print(package, version(package))

In [ ]:
from holdout_workflow import run
metrics = run(trials=20)  # Automatically separates incompatible saved results
RESULTS_DIR = Path(metrics.attrs['output_dir'])
print('Results saved to:', RESULTS_DIR)
metrics[['seed','variant','evaluation','model','roc_auc','recall','specificity']]

In [ ]:
import pandas as pd
summary = pd.read_csv(RESULTS_DIR / 'summary.csv')
summary

In [ ]:
%run scripts/build_holdout_report.py --results-dir $RESULTS_DIR

## Optional native PySpark holdout
This supplementary path runs Spark logistic regression, random forest and GBT with training-only Spark CV. It does not implement the five-family Optuna comparison above. PySpark and Java must already be available. Its preparation uses all source predictors and standardization.
For the saved Spark execution notebook, open `../PySpark/run_spark_workflow.ipynb`.


In [ ]:
# Uncomment to run the supplementary Spark benchmark in a Spark-capable environment.
# import sys
# sys.path.insert(0, str(PROJECT_ROOT.parent / 'PySpark'))
# from spark_workflow import create_spark, load_dataset, run_holdout
# spark = create_spark()
# frame, numeric, categorical = load_dataset(spark, PROJECT_ROOT/'data/raw/dataset_gallstone')
# spark_metrics = run_holdout(spark, frame, numeric, categorical,
#                             output_dir=PROJECT_ROOT.parent/'PySpark/results/spark_holdout', tune=True)